# Lakehouse to Graph

**Run locally from the project directory. Reproduces `make demo`'s load and analytics from
the Unity Catalog tables instead of `data/*.csv`.**

The same load `load.py` performs and the same algorithms `gds.py` runs, with one
substitution: the instance layer is read from the nine Unity Catalog tables rather than the
CSVs. Every step prints what it moved and the graph is queried back at the end, so the
notebook is readable as an account of how the data reaches Neo4j.

Three things land in the graph, from three different places:

| Layer | Where it comes from | What it holds |
|-------|--------------------|---------------|
| Instance | the nine Unity Catalog tables | customers, suppliers, business units, invoices, revenue entries, compliance findings, and the three bridge tables behind `SUPPLIES` and `OWNED_BY` |
| Knowledge | `generate_data`'s definitions, **graph only** | `Policy`, `Entity`, `BusinessRule`, `BusinessTerm`, `Measure`, `Threshold`, `DataSource`, `GraphMetric` |
| Graph-native scores | the three algorithms below | `Supplier.betweenness`, `Customer.pagerank`, `Customer.delinquencySimilarity`, and the two cutoffs placed from their distributions |

The middle row is the point. `guard.py` fails the build if governed vocabulary reaches Unity
Catalog, so an authored definition exists in one place only, and the demo turns on the
difference between an answer that cites one and an answer that cannot.

## What you'll learn

- Read a Unity Catalog table through the statement execution API, following chunked results
- Recover the two columns the lakehouse deliberately withholds from the invoice rows
- Derive five of the seven relationship sets from foreign keys, since only three are tables
- Check referential integrity before writing, because the loader issues `CREATE`, not `MERGE`
- Load the knowledge layer that has no lakehouse equivalent, and wire it to the instances
- Run betweenness, weighted PageRank and kNN, and place two governed thresholds from their distributions
- Walk from a score to the term, rule and threshold behind it

## This is a driver, not a second implementation

Every piece of logic is imported from `load.py`, `gds.py` or `generate_data.py` and called:
the type converters, the integrity check, the node and relationship writers, the
knowledge-layer definitions, the three algorithms, the threshold placement and the
classification write-backs. The only new code reads a warehouse table where the pipeline
reads a CSV. Nothing is restated, so nothing can drift from the pipeline it is showing.

`make demo` is still the supported way to rebuild the demo.

## Prerequisites

- The nine Unity Catalog tables exist, which means `upload.py` has run against the current `data/`
- `.env` is filled in, both the Databricks and the Neo4j halves. Section 0 reads it through `upload.read_config`
- Launched from the project directory so the modules import: `uv run --with jupyterlab jupyter lab lakehouse_to_graph.ipynb`

> **DESTRUCTIVE.** Section 6 runs `DETACH DELETE` over every node in the database named by
> `NEO4J_DATABASE`, exactly as `load.py` does. Point `.env` at a database dedicated to this
> demo.

---
## 0. Configuration

Settings come from `.env` through `upload.read_config`, so this notebook resolves them the
way the rest of the pipeline does, including the two exclusive Databricks auth modes. The
protagonists and cohorts come from `data/ground_truth.json`, which is the pipeline's own
manifest for the current build.

In [ ]:
# ============================================
# CONFIGURATION
# ============================================

import json
from datetime import datetime
from pathlib import Path

from databricks.sdk import WorkspaceClient
from dotenv import load_dotenv
from graphdatascience import GraphDataScience
from neo4j import GraphDatabase

import gds as pipeline
import generate_data
import load as csv_loader
import upload

HERE = Path.cwd()
DATA_DIR = HERE / "data"

if not (HERE / "load.py").exists():
    raise RuntimeError(f"Run this notebook from the project directory. Current: {HERE}")

load_dotenv(HERE / ".env")
cfg = upload.read_config()

# The build's manifest. evaluated_at stamps every classification edge, exactly as
# gds.py's main() derives it, so a rerun of either writes the same timestamp.
ground_truth = json.loads((DATA_DIR / "ground_truth.json").read_text())
evaluated_at = f"{ground_truth['as_of_date']}T00:00:00Z"
protags = pipeline.Protagonists.from_ground_truth(ground_truth)
near_miss = set(ground_truth["classification_cohorts"]["near_miss_customers"])

print(f"Configuration ready - {cfg.catalog}.{cfg.schema} -> {cfg.neo4j_uri}")
print(f"  warehouse   {cfg.warehouse_id}")
print(f"  database    {cfg.neo4j_database}   (WIPED in section 6)")
print(f"  as of       {ground_truth['as_of_date']}")

In [ ]:
workspace = (
    WorkspaceClient(profile=cfg.profile)
    if cfg.profile
    else WorkspaceClient(host=cfg.host, token=cfg.token)
)

# Two Neo4j handles, because the pipeline uses two: a plain driver session for the
# load, the way load.py writes, and a GraphDataScience client for the algorithms,
# the way gds.py runs them.
driver = GraphDatabase.driver(cfg.neo4j_uri, auth=cfg.neo4j_auth)
driver.verify_connectivity()
session = driver.session(database=cfg.neo4j_database)
gds = GraphDataScience(cfg.neo4j_uri, auth=cfg.neo4j_auth, database=cfg.neo4j_database)

print(f"Connected  |  warehouse {cfg.warehouse_id}  |  GDS version: {gds.version()}")

---
## 1. Helpers: reading a table, and writing what was read

The statement execution API returns every value as a string and carries the column types
separately in the response manifest, so `read_sql` hands back raw string rows and leaves
typing to `typed_rows`. Large results arrive in chunks, so it follows `next_chunk_index`
rather than trusting the first chunk to be the whole answer.

`typed_rows` applies `load.py`'s own converters and mirrors its rule: a column the
`NodeSpec` declares gets converted, every other column stays a string. A second,
independently written type map is what would make the two loaders disagree about what a
node property holds, so there is not one here.

`write_nodes` and `write_rels` wrap the pipeline's two writers with the check that matters
in a notebook, where a cell that half succeeds is easy to scroll past. Each prints
`created/total`, the same shape `aircraft-graphrag` uses: on a clean run the two match, and
a shortfall raises instead of surfacing later as a missing edge.

In [ ]:
def read_sql(statement: str) -> list[dict[str, str | None]]:
    """Run one statement on the warehouse and return rows as dicts of raw strings.

    Chunked results are followed to the end. A statement whose result is one chunk
    returns from the first call and the loop body never runs.
    """
    response = upload.run_sql(workspace, cfg, statement)
    columns = [column.name for column in response.manifest.schema.columns]
    values: list[list[str | None]] = []
    chunk = response.result
    while chunk is not None:
        values.extend(chunk.data_array or [])
        if chunk.next_chunk_index is None:
            break
        chunk = workspace.statement_execution.get_statement_result_chunk_n(
            response.statement_id, chunk.next_chunk_index
        )
    return [dict(zip(columns, row, strict=True)) for row in values]


def convert(value: str | None, kind: str | None) -> object:
    """One value, typed the way load.py types it.

    NULL from the warehouse and an empty field in a CSV both mean absent, so both
    become None and the property is left off the node.
    """
    if value is None or value == "":
        return None
    return csv_loader.CONVERTERS[kind](value) if kind else value


def typed_rows(
    raw_rows: list[dict[str, str | None]], types: dict[str, str]
) -> list[dict[str, object]]:
    """Apply a NodeSpec's type map, leaving undeclared columns as strings."""
    return [
        {key: convert(value, types.get(key)) for key, value in raw.items()}
        for raw in raw_rows
    ]


def write_nodes(label: str, rows: list[dict[str, object]]) -> None:
    """Load nodes through load.py's writer, printing created/total."""
    created = csv_loader.load_nodes(session, SPEC_BY_LABEL[label], rows)
    if created != len(rows):
        raise RuntimeError(f":{label} wrote {created} of {len(rows)} nodes.")
    print(f"  {created:>5}/{len(rows):<5} :{label}")


def write_rels(spec: csv_loader.RelSpec, rows: list[dict[str, object]]) -> None:
    """Load relationships through load.py's writer, printing created/total."""
    created = csv_loader.load_rels(session, spec, rows)
    if created != len(rows):
        raise RuntimeError(
            f"{spec.rel_type} {spec.src_label}->{spec.dst_label} wrote "
            f"{created} of {len(rows)} relationships."
        )
    print(
        f"  {created:>5}/{len(rows):<5} {spec.rel_type} "
        f"({spec.src_label} -> {spec.dst_label})"
    )


SPEC_BY_LABEL = {spec.label: spec for spec in csv_loader.NODE_SPECS}
print(f"{len(SPEC_BY_LABEL)} node specs imported from load.py")

---
## 2. The six instance node tables

One table per node label. `revenue_entries` is the only one needing a projection rather than
`SELECT *`: its `period` column holds `YYYY-MM` in the CSV and inference resolves that to a
`DATE` on the first of the month at upload, so it is formatted back here and both loaders
put the same string on the node.

> **The ordering is pinned deliberately.** A `SELECT` with no `ORDER BY` returns rows in
> whatever order the files give up, and node creation order sets the internal ids the kNN in
> section 14 samples its initial neighbourhoods from. Ordering by `id` makes this notebook
> reproducible on its own terms. It does not reproduce the CSV order exactly, because the
> generator appends the hand-named accounts after the numbered ones, so the Risky Customer
> cohort here can differ from the one a CSV load produces. The governed screen is unchanged
> and its assertions are about the cohort rather than its membership, which is what makes
> that difference reportable instead of a failure.

In [ ]:
LAKEHOUSE_TABLES = {
    "Customer": "customers",
    "Supplier": "suppliers",
    "BusinessUnit": "business_units",
    "Invoice": "invoices",
    "RevenueEntry": "revenue_entries",
    "ComplianceFinding": "compliance_findings",
}

# Projections for tables whose lakehouse shape differs from the CSV. Only
# revenue_entries needs one; every other table is read whole.
PROJECTIONS = {
    "revenue_entries": (
        "id, businessUnitId, date_format(period, 'yyyy-MM') AS period, "
        "amount, currency, reconciled"
    ),
}

node_rows: dict[str, list[dict[str, object]]] = {}
for label, table in LAKEHOUSE_TABLES.items():
    projection = PROJECTIONS.get(table, "*")
    raw = read_sql(f"SELECT {projection} FROM {upload.fqn(cfg, table)} ORDER BY id")
    node_rows[label] = typed_rows(raw, SPEC_BY_LABEL[label].types)
    columns = ", ".join(node_rows[label][0]) if node_rows[label] else "(empty)"
    print(f"  {len(node_rows[label]):>5}  :{label:<18} from {table}")
    print(f"         {columns}")

---
## 3. The two features the lakehouse withholds

`upload.py` excludes five columns from the `customers` table. Three of them, `churnRisk`,
`profitabilityTrend` and `upsellScore`, are predicted labels nothing downstream reads, and
they stay withheld here. The other two, `avgDaysLate` and `overdueShare`, are the
payment-behaviour features the kNN projects. They are withheld because they are
pre-computed judgements, not because they are secret: both are derivable from the invoice
rows, which the lakehouse carries in full.

So they are derived, by calling the generator's own `add_payment_features` on the rows
already read. Reusing that function rather than writing the aggregate in SQL is the point:
its rounding is what the fixed-seed data was shaped against, and Spark's `round` breaks
halfway cases the other way from Python's, which would move the z-scores the neighbourhoods
are computed from.

In [ ]:
# add_payment_features keys invoices by customer_id, the CSV column name. The
# lakehouse column is customerId, so the alias is added for the call.
invoices_by_generator_key = [
    {**invoice, "customer_id": invoice["customerId"]} for invoice in node_rows["Invoice"]
]
generate_data.add_payment_features(node_rows["Customer"], invoices_by_generator_key)

WITHHELD = ("churnRisk", "profitabilityTrend", "upsellScore")
sample = node_rows["Customer"][0]
print(f"derived on {len(node_rows['Customer'])} :Customer rows: avgDaysLate, overdueShare")
print(f"  {sample['id']}  avgDaysLate={sample['avgDaysLate']}  "
      f"overdueShare={sample['overdueShare']}")
print(f"still withheld, nothing downstream reads them: {', '.join(WITHHELD)}")
print(f"  present on the derived rows: {[c for c in WITHHELD if c in sample]}")

---
## 4. Relationships, read off the foreign keys

The load needs seven edge sets and Unity Catalog holds nine tables, because five of those
edges are a foreign key on a node table rather than a table of their own. An invoice carries
its `customerId`, so `HAS_INVOICE` is that column paired with the invoice id, and the same
holds for `BELONGS_TO`, `RECOGNIZES` and `HAS_FINDING`. The remaining sets are the three
bridge tables, which exist because the link is many-to-many: `supplier_business_units`,
`supply_relationships` for the supplier-to-supplier chain, and `owned_by`, the only one
carrying a property, the ownership stake the weighted PageRank reads.

The two `SUPPLIES` entries are the same relationship type with different endpoints, which is
how the pipeline models it. The supplier-to-supplier edges are the multi-tier chain, and
they fall out of the section 12 projection cleanly because the business-unit endpoint is not
a `Supplier`.

In [ ]:
# (rel_type, src_label, dst_label, table, src_col, dst_col, prop_types)
EDGE_SOURCES = [
    ("HAS_INVOICE", "Customer", "Invoice", "invoices", "customerId", "id", {}),
    ("BELONGS_TO", "Customer", "BusinessUnit", "customers", "id", "businessUnitId", {}),
    ("RECOGNIZES", "BusinessUnit", "RevenueEntry", "revenue_entries", "businessUnitId", "id", {}),
    ("SUPPLIES", "Supplier", "BusinessUnit", "supplier_business_units", "supplierId", "businessUnitId", {}),
    ("SUPPLIES", "Supplier", "Supplier", "supply_relationships", "fromSupplierId", "toSupplierId", {}),
    ("HAS_FINDING", "Customer", "ComplianceFinding", "compliance_findings", "customerId", "id", {}),
    ("OWNED_BY", "Customer", "Customer", "owned_by", "customer_id", "parent_customer_id",
     {"ownershipPct": "float"}),
]

rel_data: list[tuple[csv_loader.RelSpec, list[dict[str, object]]]] = []
for rel_type, src_label, dst_label, table, src_col, dst_col, prop_types in EDGE_SOURCES:
    projection = ", ".join([src_col, dst_col, *prop_types])
    raw = read_sql(
        f"SELECT {projection} FROM {upload.fqn(cfg, table)} ORDER BY {src_col}, {dst_col}"
    )
    rows = [
        {
            "src": row[src_col],
            "dst": row[dst_col],
            "props": {key: convert(row[key], kind) for key, kind in prop_types.items()},
        }
        for row in raw
    ]
    spec = csv_loader.RelSpec(table, rel_type, src_col, src_label, dst_col, dst_label, prop_types)
    rel_data.append((spec, rows))
    print(f"  {len(rows):>5}  {rel_type:<12} {src_label:>12} -> {dst_label:<18} from {table}")

---
## 5. Referential integrity, checked before anything is written

`load.py`'s `check_integrity` resolves every relationship endpoint against the node ids and
rejects a repeated endpoint pair. Both halves matter more than they look. The writer issues
`CREATE` rather than `MERGE`, so a duplicated pair becomes a second parallel edge, and on
the two weighted networks that is a silently doubled ownership stake or a shifted
betweenness distribution rather than an error.

Running the imported function here puts the lakehouse-sourced rows through the same check
the CSVs face, so a foreign key pointing at a row the upload dropped fails now rather than
after the database has been wiped.

In [ ]:
errors = csv_loader.check_integrity(node_rows, rel_data)
if errors:
    for error in errors:
        print(error)
    raise RuntimeError(f"{len(errors)} referential integrity errors; nothing loaded.")

total_nodes = sum(len(rows) for rows in node_rows.values())
total_rels = sum(len(rows) for _, rows in rel_data)
print(f"Check passed - {total_nodes} instance nodes and {total_rels} relationships resolve")

---
## 6. Wipe and constrain

`load.py`'s own two steps, in its order. The wipe is why the target database has to be
dedicated to this demo. Constraints come before any write because a uniqueness constraint
also creates the index every relationship write looks its endpoints up through; without them
each edge costs a full label scan.

The constraints cover the knowledge-layer labels as well as the instance ones, because they
are declared once from the same `NODE_SPECS` list.

In [ ]:
csv_loader.wipe(session)
csv_loader.create_constraints(session)

---
## 7. Write the instance layer

Nodes first, then relationships, batched with `UNWIND` by the pipeline's two writers.

In [ ]:
print("Loading instance nodes (created/total)...")
for label in LAKEHOUSE_TABLES:
    write_nodes(label, node_rows[label])

print("\nLoading instance relationships (created/total)...")
for spec, rows in rel_data:
    write_rels(spec, rows)

---
## 8. The knowledge layer, which the lakehouse does not hold

Everything above came out of Unity Catalog. Nothing below can, and that is the reason a
graph is in this demo at all.

The `Policy`, `Entity`, `BusinessRule`, `BusinessTerm`, `Measure`, `Threshold`, `DataSource`
and `GraphMetric` nodes have no lakehouse equivalent, and `guard.py` fails the build if
governed vocabulary shows up in Unity Catalog. So they are loaded from the generator's
module-level definitions, which are the same literals the CSVs are written from.
`generate_data` is import-safe: everything below its main guard is data.

Two thresholds arrive without a value. `THR-03` and `THR-04` govern metrics that do not
exist until the algorithms have run, so their cutoffs are resolved from the computed
distributions in section 15. `THR-05` is authored rather than fitted, because a neighbour
share is already on the scale of the metric it governs, so it carries its value from the
start and section 15 only verifies it.

In [ ]:
KNOWLEDGE_NODES = [
    ("Entity", generate_data.ENTITIES),
    ("BusinessTerm", generate_data.BUSINESS_TERMS),
    ("BusinessRule", generate_data.BUSINESS_RULES),
    ("Measure", generate_data.MEASURES),
    ("GraphMetric", generate_data.GRAPH_METRICS),
    ("Policy", generate_data.POLICIES),
    ("Threshold", generate_data.THRESHOLDS),
    ("DataSource", generate_data.DATA_SOURCES),
]

# (rel_type, src_label, dst_label, src_key, dst_key, rows)
KNOWLEDGE_EDGES = [
    ("DEFINED_BY", "BusinessTerm", "BusinessRule", "term_id", "rule_id", generate_data.DEFINED_BY),
    ("DEFINED_BY", "Measure", "BusinessRule", "measure_id", "rule_id", generate_data.MEASURE_DEFINED_BY),
    ("MEASURED_BY", "BusinessTerm", "Measure", "term_id", "measure_id", generate_data.MEASURED_BY),
    ("SCORED_BY", "BusinessTerm", "GraphMetric", "term_id", "metric_id", generate_data.SCORED_BY),
    ("USES_THRESHOLD", "BusinessRule", "Threshold", "rule_id", "threshold_id", generate_data.RULE_THRESHOLDS),
    ("EVALUATES", "BusinessRule", "Entity", "rule_id", "entity_id", generate_data.EVALUATES),
    ("CONSTRAINS", "Policy", "Entity", "policy_id", "entity_id", generate_data.CONSTRAINS),
    ("GOVERNS", "Policy", "BusinessRule", "policy_id", "rule_id", generate_data.GOVERNS),
    ("APPLIES_TO", "Threshold", "BusinessTerm", "threshold_id", "term_id", generate_data.APPLIES_TO),
    ("MAPS_TO", "Entity", "DataSource", "entity_id", "data_source_id", generate_data.MAPS_TO),
]


def knowledge_rows(items: list[dict]) -> list[dict[str, object]]:
    """Normalize the generator's literals into uniform node rows.

    The definitions omit a key where a CSV would carry an empty column, so the key
    union is filled in. An absent value and a None both leave the property off the
    node, which is what an empty CSV column does too.
    """
    keys = {key for item in items for key in item}
    return [
        {key: (None if item.get(key) == "" else item.get(key)) for key in keys}
        for item in items
    ]


print("Loading knowledge-layer nodes (created/total)...")
for label, items in KNOWLEDGE_NODES:
    write_nodes(label, knowledge_rows(items))

print("\nLoading knowledge-layer relationships (created/total)...")
for rel_type, src_label, dst_label, src_key, dst_key, items in KNOWLEDGE_EDGES:
    rows = [{"src": item[src_key], "dst": item[dst_key], "props": {}} for item in items]
    spec = csv_loader.RelSpec(
        "generate_data", rel_type, src_key, src_label, dst_key, dst_label
    )
    write_rels(spec, rows)

---
## 9. Wiring the knowledge layer to the instances

`REALIZED_AS` runs from a logical `Entity` to every instance node that realizes it, and it
is the edge that makes the graph answerable in both directions. `MAPS_TO` already points an
entity at the Unity Catalog table behind it, so from any instance node a walk reaches both
the definition that governs it and the column that produced it.

The mapping needs no source file, because an `Entity` is named after the label it describes.
`SupplyRelationship` is the one entity that gets no edges, since the thing it describes is a
relationship rather than a node, which is why it appears in the entity list and never in
this output.

In [ ]:
# Entity name to node label, derived rather than listed: an Entity is named after
# the label it describes. SupplyRelationship names a relationship, so it matches
# nothing here and correctly receives no REALIZED_AS edges.
realized = {
    entity["id"]: entity["name"]
    for entity in generate_data.ENTITIES
    if entity["name"] in LAKEHOUSE_TABLES
}

print("Loading REALIZED_AS (created/total)...")
for entity_id, label in realized.items():
    rows = [
        {"src": entity_id, "dst": row["id"], "props": {}} for row in node_rows[label]
    ]
    spec = csv_loader.RelSpec(
        entity_id, "REALIZED_AS", "entity_id", "Entity", "instance_id", label
    )
    write_rels(spec, rows)

skipped = [e["name"] for e in generate_data.ENTITIES if e["id"] not in realized]
print(f"\nno node instances, relationship entities: {', '.join(skipped)}")

---
## 10. Classifications, derived from the rules

Four business terms are findable from lakehouse columns, and three of them are derived here
by applying the rule rather than by copying a list:

| Term | Rule | Derivation |
|------|------|------------|
| Defaulted Customer | `RULE-02` | `defaultedPeriod` is set |
| Delinquent Customer | `RULE-03` | the last three invoices each more than the Late Payment Threshold days late, through the generator's own `compute_delinquent` |
| High-Risk Supplier | `RULE-04` | `riskScore` at or above the Supplier Risk Threshold |

**Strategic Account is the one that cannot be derived.** `RULE-01` reads a platinum segment
and a strategic flag from account management, and no lakehouse column carries that flag: it
is a judgement, not a measurement. It is read from `ground_truth.json`. Leaving it out was
the other option and it is worse, because a term with a definition and no instances is the
exact discovery failure this project already found once: an agent learns the classification
pattern from the terms that have edges, applies it to the term that has none, gets an empty
result and truthfully reports that the system does not classify them.

The three graph-native terms are absent from this cell on purpose. Their cohorts do not
exist until the algorithms have scored the networks, so `gds.py` writes them in sections 12
and 14, and Ownership Risk stays a live traversal.

In [ ]:
# Reason strings mirror the generator's, so the provenance on an edge reads the
# same whichever loader wrote it.
DERIVED_TERMS = [
    (
        "TERM-01",
        "Customer",
        ground_truth["classification_cohorts"]["strategic_accounts"],
        "platinum segment and flagged strategic by account management",
    ),
    (
        "TERM-02",
        "Customer",
        sorted(c["id"] for c in node_rows["Customer"] if c["defaultedPeriod"]),
        "default period recorded in the snapshot",
    ),
    (
        "TERM-03",
        "Customer",
        generate_data.compute_delinquent(node_rows["Customer"], invoices_by_generator_key),
        f"each of the last three invoices more than "
        f"{generate_data.LATE_DAYS_THRESHOLD} days late",
    ),
    (
        "TERM-04",
        "Supplier",
        [
            s["id"]
            for s in node_rows["Supplier"]
            if s["riskScore"] >= generate_data.SUPPLIER_RISK_THRESHOLD
        ],
        "risk score at or above the supplier risk threshold",
    ),
]

print("Loading CLASSIFIED_AS (created/total)...")
for term_id, label, ids, reason in DERIVED_TERMS:
    rows = [
        {
            "src": node_id,
            "dst": term_id,
            "props": {
                "reason": reason,
                "evaluatedAt": datetime.fromisoformat(evaluated_at),
                "ruleVersion": generate_data.RULE_VERSION,
            },
        }
        for node_id in ids
    ]
    source = "ground_truth.json" if term_id == "TERM-01" else "derived from the rule"
    spec = csv_loader.RelSpec(
        f"{term_id} ({source})", "CLASSIFIED_AS", "entity_id", label, "term_id", "BusinessTerm"
    )
    write_rels(spec, rows)
    print(f"         {term_id} {source}")

---
## 11. What landed

The load is done, so this is the graph the algorithms will run on. Counting it back out of
Neo4j rather than reporting what was sent is the point: a count taken from the database is
the only one that can disagree with the loader, and a disagreement is what you would want to
see.

In [ ]:
def show(query: str, title: str) -> None:
    print(title)
    for record in session.run(query):
        key, count = record.values()
        print(f"  {count:>6}  {key}")


show(
    "MATCH (n) UNWIND labels(n) AS label "
    "RETURN label, count(*) AS nodes ORDER BY nodes DESC, label",
    "Node counts by label:",
)
print()
show(
    "MATCH ()-[r]->() RETURN type(r) AS rel, count(*) AS rels ORDER BY rels DESC, rel",
    "Relationship counts by type:",
)

---
## 12. Algorithm 1: supplier betweenness

From here the notebook calls `gds.py`'s functions in the order its `main` calls them, one
heading per algorithm, so the run reads as a sequence rather than a log. Their output is the
pipeline's output, including the assertions that stop the run.

| Algorithm | Property written | What it measures |
|-----------|-----------------|------------------|
| **Betweenness** | `Supplier.betweenness` | how many multi-tier supply paths run through one supplier, which is position rather than partner count |
| **Weighted PageRank** | `Customer.pagerank` | how much failure reaches a customer through the stakes held in it, seeded on the defaulted accounts |
| **kNN on payment behaviour** | `Customer.delinquencySimilarity` | the share of a customer's nearest neighbours in payment-behaviour space already classified Delinquent |

Betweenness answers a question about position. The projection keeps `Supplier` nodes and
`SUPPLIES` edges undirected, and the supplier-to-business-unit edges drop out because their
far endpoint is not a `Supplier`, so what is projected is the raw-material chain. A supplier
scores by how many supply paths run through it, which is not the same measure as how many
partners it has, and `report_degree_overlap` prints how far the two rankings agree.

`concentration_cutoff` then resolves `THR-03`. The percentile is governed and fixed before
any score exists; resolving it against the distribution this run produced is the only part
that could not be known in advance, and how many suppliers land in the cohort is an output
rather than a decision.

In [ ]:
betweenness = pipeline.compute_betweenness(gds, protags)
conc = pipeline.concentration_cutoff(betweenness)
pipeline.assert_betweenness(betweenness, conc, protags)
pipeline.report_degree_overlap(gds, betweenness, protags)
pipeline.write_betweenness(gds, betweenness)
pipeline.write_critical_supplier_labels(gds, conc, evaluated_at)

---
## 13. Algorithm 2: weighted ownership PageRank

Personalized PageRank over the ownership network, seeded on every defaulted customer in the
book and weighted by the size of each stake. The weight is what makes the answer a graph
result rather than a hop count: a default next door held through a token stake transmits
almost nothing, while a controlling chain several levels long transmits a great deal, so
nearness on its own decides nothing.

`trading_customers` then narrows the ranking to the accounts the Ownership Risk term can
apply to at all. The raw top of the distribution is the defaulted customers and the holding
companies that own them, which is arithmetic rather than a finding.

Convergence is checked before the scores are used, because the cutoff is read off them to
six decimal places.

In [ ]:
pagerank = pipeline.compute_pagerank(gds, protags)
trading = pipeline.trading_customers(gds)
pipeline.print_top_trading(pagerank, protags, trading)
cont = pipeline.contagion_cutoff(pagerank, protags, trading)
pipeline.assert_pagerank(pagerank, cont, protags, trading)
pipeline.write_pagerank(gds, pagerank)

---
## 14. Algorithm 3: payment-behaviour kNN

The two features derived in section 3 come into play here. The projection carries
`avgDaysLate` and `overdueShare`, scales the pair to z-scores as one vector, and asks each
customer who its nearest neighbours are in that space. Scaling first is not a tidy-up: on
the raw columns the feature with the narrower range would decide the neighbourhood on its
own.

The metric is then the share of those neighbours already classified Delinquent, so a
customer scores by the company its payment behaviour keeps rather than by a threshold on its
own columns. Neighbourhoods are computed over every customer, because the delinquent
accounts have to be in the candidate set for anything to be near them, while
`risky_candidates` decides separately who the resulting classification may apply to.

`write_similarity_edges` then materializes the evidence as `SIMILAR_PAYMENT_BEHAVIOR` edges,
so the answer on screen is a list of real accounts rather than a decimal.

This is where the ordering note from section 2 lands. The cohort is an output and its
membership can differ from a CSV load's. What `assert_risky_customers` requires is that the
screen catches a cohort rather than a single name, that nobody in it has already failed, and
that the planted near-miss customers are findable.

In [ ]:
knn = pipeline.compute_delinquency_similarity(gds)
candidates = pipeline.risky_candidates(gds)
screen = pipeline.risky_screen(knn.scores, candidates)
pipeline.assert_risky_customers(screen, near_miss, candidates)
pipeline.write_delinquency_similarity(gds, knn.scores)
pipeline.write_similarity_edges(gds, screen, knn.delinquent_links, evaluated_at)
pipeline.write_risky_customer_labels(gds, screen, evaluated_at)

---
## 15. Thresholds and governed vocabulary

The two cutoffs the algorithms produced go onto the `Threshold` nodes section 8 left without
a value, and onto their rules as an inline property, so the graph-native rules carry their
number the way the column-findable ones already do. `THR-05` is verified rather than
written, because it was authored before any similarity was computed.

`write_governed_terms` then names the governing term on every node carrying a metric. That
property says which term governs the score, not which nodes qualify under it, so every
supplier carries the same string whether or not it cleared the cutoff. It is what carries
the governed vocabulary into any result set that touches the node.

> **Unlike `gds.py`, this does not write the resolved cutoffs back to `data/thresholds.csv`.**
> The pipeline does that so a later CSV reload carries them. Writing them from here would
> edit the pipeline's inputs as a side effect of a notebook run, and the graph this notebook
> built already holds the values.

In [ ]:
pipeline.header("Graph-native thresholds (set from the computed distributions)")
cutoffs = {
    pipeline.CONCENTRATION_THRESHOLD_ID: conc.value,
    pipeline.CONTAGION_THRESHOLD_ID: cont.value,
}
pipeline.write_thresholds(gds, cutoffs)
pipeline.write_rule_thresholds(
    gds,
    {
        pipeline.CONCENTRATION_RULE_ID: conc.value,
        pipeline.CONTAGION_RULE_ID: cont.value,
    },
)
pipeline.check_governed_threshold(gds)

pipeline.header("Governed vocabulary bound to the metrics (Neo4j only)")
pipeline.write_governed_terms(gds)

---
## 16. Verify the graph answers

The load and the algorithms both reported success, which says the writes landed and says
nothing about whether the result is reachable. `check_three_legs` and
`check_risky_customer_legs` walk the graph the way a question would: from a score to the
term that governs it, to the rule that defines it, to the threshold it is compared against,
and to the classification and its evidence.

That is the check an earlier version of this demo did not have, and its absence is how a
correct, fully wired knowledge layer still produced an empty first result.

In [ ]:
pipeline.check_three_legs(gds, protags)
pipeline.check_risky_customer_legs(gds, screen)

---
## 17. Where the two views meet

The join between the lakehouse and the graph is the `id` string, shared by construction: a
`customers` row and its `Customer` node carry the same value, and so does every other pair.
That is what makes a cross-source answer possible without a key mapping. Take the id set
from whichever side owns the structure, and look the figures up on the other.

The query below reads the Critical Supplier cohort out of the graph, which is the only place
it exists, then prints those ids against the Unity Catalog table the suppliers came from.
The graph decided who is in the set. The lakehouse still holds every column about them.

In [ ]:
cohort = gds.run_cypher(
    """
    MATCH (s:Supplier)-[r:CLASSIFIED_AS]->(t:BusinessTerm {name: 'Critical Supplier'})
    RETURN s.id AS id, s.name AS name, s.betweenness AS betweenness,
           s.betweennessGovernedTerm AS governedTerm, r.reason AS reason
    ORDER BY betweenness DESC, id
    """
)
print("Critical Supplier cohort, resolved in the graph:")
print(cohort.to_string(index=False))

ids = ", ".join(f"'{node_id}'" for node_id in cohort["id"])
lakehouse = read_sql(
    f"SELECT id, name, category, subcategory, riskScore "
    f"FROM {upload.fqn(cfg, 'suppliers')} WHERE id IN ({ids}) ORDER BY id"
)
print(f"\nThe same suppliers in {cfg.schema}.suppliers, joined on id:")
for row in lakehouse:
    print(f"  {row['id']}  {row['name']:<26} {row['subcategory']:<20} "
          f"riskScore={row['riskScore']}")
print("\nNo column in that table carries the cohort. It exists only where it was resolved.")

---
## 18. Close the connections

In [ ]:
session.close()
driver.close()
gds.close()
print("Connections closed.")

---
## What we just did

Nine Unity Catalog tables became a property graph: six node labels, seven relationship
types, the two payment-behaviour features recovered from the invoice rows, and five of the
seven edge sets read straight off foreign keys. Then the knowledge layer went in on top,
from the generator's definitions rather than from any table, and three algorithms scored the
two networks and placed two governed thresholds from their own distributions.

The graph now holds what `make demo` produces, with two differences worth stating plainly
rather than leaving for someone to find:

- The three withheld predicted-label columns are not on the `Customer` nodes, because the lakehouse does not carry them. Nothing in the demo reads them.
- The resolved cutoffs are in the graph but not written back to `data/thresholds.csv`, so a later `uv run load.py` from the CSVs would load the previous build's values. Run `make demo` if you need the two in step.

**Next** Run `uv run expected_results.py` for the presenter's figures for this build, and
`uv run guard.py` to confirm no governed vocabulary reached Unity Catalog.

---
## Neo4j console queries

The three legs behind one Critical Supplier: the score, the term that governs it, the rule
that defines it, and the threshold it was compared against.

```cypher
MATCH (s:Supplier)-[:CLASSIFIED_AS]->(t:BusinessTerm {name: 'Critical Supplier'})
MATCH (t)-[:DEFINED_BY]->(r:BusinessRule)-[:USES_THRESHOLD]->(th:Threshold)
MATCH (t)-[:SCORED_BY]->(m:GraphMetric)
RETURN s.id, s.name, s.betweenness, t.name, r.expression, th.name, th.value, m.algorithm
ORDER BY s.betweenness DESC
```

The round trip a graph makes possible and a table does not: from an instance node out to the
definition that governs it, and back down to the lakehouse column that produced it.

```cypher
MATCH (e:Entity)-[:REALIZED_AS]->(c:Customer)
MATCH (e)-[:MAPS_TO]->(ds:DataSource)
MATCH (rule:BusinessRule)-[:EVALUATES]->(e)
RETURN e.name, ds.system, ds.table, collect(DISTINCT rule.name) AS rules
LIMIT 5
```

The multi-tier supply chain the betweenness scores, drawn from the top of the ranking.

```cypher
MATCH path = (:Supplier)-[:SUPPLIES*2..4]->(s:Supplier)-[:SUPPLIES]->(bu:BusinessUnit)
WHERE s.betweenness IS NOT NULL
RETURN path
LIMIT 50
```

The evidence behind one Risky Customer: the already-delinquent accounts whose payment
behaviour it most resembles.

```cypher
MATCH (c:Customer)-[:CLASSIFIED_AS]->(:BusinessTerm {name: 'Risky Customer'})
MATCH (c)-[sim:SIMILAR_PAYMENT_BEHAVIOR]->(n:Customer)
RETURN c.id, c.delinquencySimilarity, n.id, sim.similarity, sim.neighbourRank
ORDER BY c.id, sim.neighbourRank
```